# 02 — Baselines

**Goal:** before tuning anything, find out how much each part of the setup is worth: trivial rules,
which feature blocks carry signal, and how the six model families compare with **library-default
hyperparameters**.

**Protocol** (same as the pipeline, via `src/fjd`):
- Scores are **PR-AUC on the fraud class** (average precision). A random ranking scores the fraud
  rate, about 0.045.
- Comparisons use **5-fold `StratifiedGroupKFold` on `group_id`** over the train split, so a
  template or company never sits on both sides of a fold. The featurizer is refit inside each fold.
- The val split gets one check at the end. **The test split is not touched.**

Runs are logged to MLflow (tag `source=02_baselines`): DagsHub if `MLFLOW_TRACKING_URI` is set,
otherwise the local `mlflow.db`.

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from fjd.data.io import read_split
from fjd.data.schema import TARGET
from fjd.features.build import build_featurizer
from fjd.models.classical import MODELS, build_model
from fjd.models.cv import cross_val_pr_auc, group_folds
from fjd.models.metrics import fraud_scores, pr_auc, pr_curve
from fjd.utils.config import load_params
from fjd.utils.plotting import INK_SECONDARY, SERIES, set_style
from fjd.utils.seed import set_seed
from fjd.utils.tracking import run_tags, setup_mlflow

set_style()
params = load_params()
SEED, N_FOLDS = params["seed"], params["tuning"]["cv_folds"]
set_seed(SEED)
print("MLflow:", setup_mlflow())

train, val = read_split("train"), read_split("val")
base_rate = train[TARGET].mean()
print(f"train {len(train):,} rows ({base_rate:.2%} fraud), val {len(val):,} rows")

## 1. Trivial baselines

Single-flag rules from the EDA, scored on the train split (no fitting involved). Any model has to
clearly beat these to be worth its complexity.

In [ ]:
rules = {
    "random (fraud rate)": np.full(len(train), base_rate),
    "no company logo": 1 - train["has_company_logo"],
    "no company profile": train["company_profile"].isna().astype(int),
    "no logo + no profile": (1 - train["has_company_logo"]) + train["company_profile"].isna(),
}
rule_scores = pd.Series({name: pr_auc(train[TARGET], s) for name, s in rules.items()})
rule_scores.to_frame("PR-AUC (train)").style.format("{:.3f}")

## 2. Which feature blocks carry signal?

Logistic regression (defaults, balanced class weights) on each block alone, then on everything.
Folds are built once and reused for every comparison below.

In [ ]:
folds = group_folds(train, params["features"], N_FOLDS, SEED)
print(f"{len(folds)} folds, {folds[0].X_train.shape[1]:,} features in fold 0")


def block_folds(prefixes: tuple[str, ...]) -> list:
    out = []
    for f in folds:
        cols = np.flatnonzero(pd.Series(f.feature_names).str.startswith(prefixes))
        out.append(
            replace(
                f,
                X_train=f.X_train[:, cols],
                X_val=f.X_val[:, cols],
                feature_names=f.feature_names[cols],
            )
        )
    return out


blocks = {
    "text (TF-IDF)": ("text__",),
    "tabular flags + counts": ("tabular__",),
    "categorical one-hot": ("categorical__",),
    "tabular + categorical": ("tabular__", "categorical__"),
    "all features": ("text__", "tabular__", "categorical__"),
}
ablation = pd.DataFrame(
    {name: cross_val_pr_auc("logreg", {}, block_folds(p), SEED) for name, p in blocks.items()}
).T
ablation.columns = [f"fold {i}" for i in range(N_FOLDS)]
ablation.insert(0, "mean", ablation.mean(axis=1))
ablation.style.format("{:.3f}").background_gradient(subset="mean", cmap="Blues")

## 3. Six model families with default hyperparameters

Library defaults plus the fixed settings in `src/fjd/models/classical.py` (class weights, seeds;
tree models first keep the 5,000 columns with the highest χ² score). One MLflow run per family.

In [ ]:
cv_results = {}
with mlflow.start_run(run_name="02_baselines", tags=run_tags("02_baselines")):
    for name in MODELS:
        scores = cross_val_pr_auc(name, {}, folds, SEED)
        cv_results[name] = scores
        with mlflow.start_run(
            run_name=f"baseline_{name}",
            nested=True,
            tags=run_tags("02_baselines") | {"model": name},
        ):
            mlflow.log_param("model_name", name)
            mlflow.log_metrics(
                {"cv_pr_auc": float(np.mean(scores)), "cv_pr_auc_std": float(np.std(scores))}
            )
        print(f"{name:14s} CV PR-AUC {np.mean(scores):.3f} ± {np.std(scores):.3f}")

cv_table = pd.DataFrame(cv_results).T
cv_table.columns = [f"fold {i}" for i in range(N_FOLDS)]
cv_table.insert(0, "std", cv_table.std(axis=1))
cv_table.insert(0, "mean", cv_table.drop(columns="std").mean(axis=1))
cv_table = cv_table.sort_values("mean", ascending=False)
cv_table.style.format("{:.3f}").background_gradient(subset="mean", cmap="Blues")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
order = cv_table.index[::-1]
for y, name in enumerate(order):
    ax.scatter(cv_results[name], [y] * N_FOLDS, color=SERIES[0], alpha=0.45, s=22, zorder=2)
    ax.scatter(np.mean(cv_results[name]), y, color=SERIES[1], marker="D", s=42, zorder=3)
ax.axvline(base_rate, color=INK_SECONDARY, lw=1, ls="--")
ax.set_yticks(range(len(order)), order)
ax.set_xlabel("PR-AUC (dots = folds, diamond = mean, dashed = random)")
ax.set_title("Default hyperparameters, group-aware 5-fold CV")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 4. One check on val

Each family is fit on the whole train split and scored once on val. This confirms the CV
ranking on data from groups the model has never seen. It is not used to choose anything.

In [ ]:
featurizer = build_featurizer(params["features"]).fit(train)
X_train, X_val = featurizer.transform(train), featurizer.transform(val)
val_scores = {
    name: fraud_scores(build_model(name, {}, SEED).fit(X_train, train[TARGET]), X_val)
    for name in MODELS
}
cv_table["val PR-AUC"] = pd.Series({n: pr_auc(val[TARGET], s) for n, s in val_scores.items()})
cv_table[["mean", "std", "val PR-AUC"]].style.format("{:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for color, name in zip(SERIES, cv_table.index, strict=False):
    curve = pr_curve(val[TARGET], val_scores[name])
    auc = cv_table.loc[name, "val PR-AUC"]
    ax.plot(
        curve["recall"],
        curve["precision"],
        color=color,
        drawstyle="steps",
        label=f"{name} ({auc:.3f})",
    )
ax.axhline(val[TARGET].mean(), color=INK_SECONDARY, lw=1, ls="--")
ax.set(xlabel="recall", ylabel="precision", xlim=(0, 1), ylim=(0, 1.02))
ax.set_title("Precision-recall on val, default hyperparameters")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 5. Takeaways

Numbers from the run on 2026-10-19 (seed 42); re-running reproduces them exactly.

1. **Rules are weak:** the best flag rule (no logo + no company profile) reaches PR-AUC 0.13, about
   3× random (0.045). Every model except Naive Bayes is about 4× better than that.
2. **Text carries most of the signal:** TF-IDF alone gives logistic regression 0.51 CV PR-AUC, and
   tabular + categorical features alone give 0.29. Adding them to the text did *not* help default
   logistic regression (0.49): ~180 tabular and one-hot columns barely register next to 50k TF-IDF
   columns at `C=1`. The tree models, which can split on them directly, are where they should pay
   off, and tuning `C` may change the picture for the linear models.
3. **Default ranking (CV mean):** LinearSVM 0.56 > LightGBM 0.52 ≈ XGBoost 0.52 > LogReg 0.49 >
   Random forest 0.44 > Naive Bayes 0.30.
4. **Folds disagree a lot:** fold standard deviations are 0.12–0.19, and fold 3 is hard for
   every model (0.11–0.32). It presumably holds fraud groups unlike anything in the other folds,
   which is exactly the "new scam campaign" case the group split is designed to measure.
   Differences under about 0.05 between models are within noise.
5. **One val split is noisy too:** val has only 102 fraud rows. Logistic regression scores 0.70 on
   val but 0.49 in CV. Hence tuning (notebook 03) optimizes the CV mean, and val is kept for
   threshold tuning and a final sanity check.